In [92]:
import os
import sys
import certifi
from dotenv import load_dotenv
# Add project root to path so we can import mcp_client
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))
load_dotenv()
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
from typing import Any, TypedDict, Annotated
import operator
import uuid
import asyncio
import json
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI
from mcp_client import (           # ← plain import now works
    tavily_mcp_search,
    aviation_mcp_call,
    extract_destination,
    forecast_mcp_search,
    weather_mcp_search,
)

In [93]:
from pathlib import Path

# Explicitly point to the project root .env
dotenv_path = Path(__file__ if '__file__' in dir() else '.').resolve().parent.parent / '.env'
# Or just hardcode it to be safe:
load_dotenv(r"D:\PIS\youtube\TripMate-Multi-Agent-AI-System\.env", override=True)

True

In [94]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TYPESAFE_API_KEY=os.getenv("TYPESAFE_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Please add it to your .env file.")

# =========================
# LLM
# =========================
llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY,
)


In [126]:
# =========================
# State - original fields kept, new control fields added
# =========================
class TravelState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], operator.add]
    user_query: str

    # Supervisor + guardrail state
    guardrail_allowed: bool
    guardrail_reason: str
    selected_agents: list[str]
    trip_constraints: dict[str, Any]
    supervisor_reasoning: str

    # Original specialist results
    flight_results: str
    hotel_results: str
    weather_results: str
    itinerary: str

    # New budget + HITL state
    budget_results: str
    approval_request: str
    approved: bool
    human_feedback: str
    final_response: str

    llm_calls: int

In [96]:
# =========================
# Shared helpers
# =========================
KNOWN_AGENTS = {
    "flight_agent",
    "hotel_agent",
    "weather_agent",
    "budget_agent",
    "itinerary_agent",
}

AGENT_ORDER = [
    "flight_agent",
    "hotel_agent",
    "weather_agent",
    "budget_agent",
    "itinerary_agent",
]

In [97]:
def _llm_text(system_prompt: str, user_prompt: str) -> str:
    response = llm.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ]
    )
    return str(response.content)


In [98]:

def _json_from_llm(text: str) -> dict[str, Any]:
    """Extract the first complete JSON object returned by the model."""
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end < start:
        raise ValueError("The model did not return a JSON object.")

    return json.loads(text[start : end + 1])


In [99]:
def _empty_constraints() -> dict[str, Any]:
    return {
        "destination": "",
        "origin": "",
        "duration": "",
        "budget": "",
        "travel_style": "",
        "special_preferences": [],
    }


In [100]:
from langchain_typesafe import Choice, Noul, Score, TypeSafeClassifier


In [119]:
from langchain_core.output_parsers import string
Jev_classifier = TypeSafeClassifier(
 model="jev-latest",
 api_key=os.environ["TYPESAFE_API_KEY"]
)

def jev_response(user_query: str , guardrail_prompt: str):

    response = Jev_classifier.invoke(
        {
            "state": (
                user_query,
            ),
            "questions": {
                "guardrail_Approval": Noul(instructions=guardrail_prompt)
            },
        }
    )

    print(response.nouls["guardrail_Approval"].noul)
    return response.nouls["guardrail_Approval"].noul >= 75



In [ ]:
# jev_response(user_query, guardrail_prompt)

0.04


False

In [ ]:
def guardrail_component(state: TravelState):
    query = state["user_query"]
    llm_calls = state.get("llm_calls", 0)

    guardrail_prompt = """Determine whether the following request belongs to travel planning or travel
information. Valid requests can include destinations, flights, hotels, weather,
budgets, visas, transportation, sightseeing, food, packing, or itineraries.

Block clearly unrelated requests and requests asking for harmful or illegal
instructions. Do not block a valid travel request merely because some details
are missing. You need to classify if it is a valid request or not."""

    allowed = jev_response(query, guardrail_prompt)
    if allowed:
        reason = "approved"
    else:
        reason = """TripMate AI can only help with travel-planning requests.
            Please ask about a destination, flight, hotel, weather, budget,
            or itinerary."""

    return {
        "guardrail_allowed": allowed,  
        "guardrail_reason": reason,
        "llm_calls": llm_calls + 1,    
    }
